In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
import xgboost as xgb
import warnings
warnings.filterwarnings("ignore")

RANDOM_STATE = 42

In [2]:
df = pd.read_csv("cleaned_shill_bidding_dataset.csv")

In [3]:
id_cols = ["Record_ID", "Auction_ID", "Bidder_ID"]
df = df.drop(columns=[c for c in id_cols if c in df.columns])

X = df.drop(columns=["Class"])
y = df["Class"]

print("Features used:", list(X.columns))
print("Feature matrix shape:", X.shape)

Features used: ['Bidder_Tendency', 'Bidding_Ratio', 'Successive_Outbidding', 'Last_Bidding', 'Auction_Bids', 'Starting_Price_Average', 'Early_Bidding', 'Winning_Ratio', 'Auction_Duration']
Feature matrix shape: (6321, 9)


In [4]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=RANDOM_STATE
)

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.1765,
    stratify=y_temp, random_state=RANDOM_STATE
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)
print("Train class ratio:\n", y_train.value_counts(normalize=True))
print("Val class ratio:\n", y_val.value_counts(normalize=True))
print("Test class ratio:\n", y_test.value_counts(normalize=True))

Train: (4423, 9) Val: (949, 9) Test: (949, 9)
Train class ratio:
 Class
0    0.893059
1    0.106941
Name: proportion, dtype: float64
Val class ratio:
 Class
0    0.893572
1    0.106428
Name: proportion, dtype: float64
Test class ratio:
 Class
0    0.893572
1    0.106428
Name: proportion, dtype: float64


In [5]:
neg, pos = np.bincount(y_train)
scale_pos_weight = neg / pos
print(f"scale_pos_weight (train-derived): {scale_pos_weight:.3f}")

scale_pos_weight (train-derived): 8.351


In [6]:
final_model = xgb.XGBClassifier(
    objective="binary:logistic",
    eval_metric="auc",
    scale_pos_weight=scale_pos_weight,
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.7,
    colsample_bytree=0.5,
    colsample_bylevel=0.5,
    min_child_weight=3,
    reg_alpha=1.0,
    reg_lambda=2.0,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

final_model.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

from sklearn.metrics import precision_recall_curve, f1_score

probs = final_model.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, probs)

f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print(f"Best threshold: {best_threshold:.3f}, F1: {f1_scores[best_idx]:.4f}")

# Use test probabilities from THIS model too, not a stale variable
test_preds_proba = final_model.predict_proba(X_test)[:, 1]
test_preds_tuned = (test_preds_proba >= best_threshold).astype(int)
print(classification_report(y_test, test_preds_tuned, digits=4))

Best threshold: 0.848, F1: 0.9806
              precision    recall  f1-score   support

           0     0.9988    0.9965    0.9976       848
           1     0.9709    0.9901    0.9804       101

    accuracy                         0.9958       949
   macro avg     0.9848    0.9933    0.9890       949
weighted avg     0.9958    0.9958    0.9958       949



In [7]:
from sklearn.calibration import CalibratedClassifierCV

calibrated_model = CalibratedClassifierCV(final_model, method="isotonic", cv=5)
calibrated_model.fit(X_train, y_train)

calibrated_probs = calibrated_model.predict_proba(X_test)[:, 1]

In [8]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before:", y_train.value_counts().to_dict())
print("After:", pd.Series(y_train_sm).value_counts().to_dict())


model_smote = xgb.XGBClassifier(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.6,
    random_state=RANDOM_STATE, n_jobs=-1
)
model_smote.fit(X_train_sm, y_train_sm)

Before: {0: 3950, 1: 473}
After: {0: 3950, 1: 3950}


,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.6
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'auc'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [9]:
from skopt import BayesSearchCV
from skopt.space import Real, Integer

search_space = {
    "max_depth": Integer(3, 7),
    "learning_rate": Real(0.01, 0.2, prior="log-uniform"),
    "subsample": Real(0.6, 1.0),
    "colsample_bytree": Real(0.4, 0.8),   # capped lower to limit single-feature dominance
    "min_child_weight": Integer(1, 10),
    "reg_alpha": Real(0, 5),
    "reg_lambda": Real(0, 5),
}

opt = BayesSearchCV(
    estimator=xgb.XGBClassifier(
        objective="binary:logistic", eval_metric="aucpr",
        scale_pos_weight=scale_pos_weight,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    search_spaces=search_space,
    n_iter=30,
    scoring="average_precision",   # PR-AUC, not ROC-AUC
    cv=5,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
opt.fit(X_train, y_train)
print(opt.best_params_, opt.best_score_)

OrderedDict({'colsample_bytree': 0.6168990620546355, 'learning_rate': 0.2, 'max_depth': 7, 'min_child_weight': 1, 'reg_alpha': 0.0, 'reg_lambda': 5.0, 'subsample': 1.0}) 0.9939356728032234


In [10]:
df["bid_intensity"] = df["Auction_Bids"] / (df["Auction_Duration"] + 1e-5)
df["price_vs_starting"] = df["Winning_Ratio"] / (df["Starting_Price_Average"] + 1e-5)
df["tendency_x_ratio"] = df["Bidder_Tendency"] * df["Bidding_Ratio"]

In [11]:
best_params = opt.best_params_   

model_tuned = xgb.XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE, n_jobs=-1,
    **best_params
)
model_tuned.fit(X_train, y_train, eval_set=[(X_val, y_val)], verbose=False)

probs_val_tuned = model_tuned.predict_proba(X_val)[:, 1]
precisions, recalls, thresholds = precision_recall_curve(y_val, probs_val_tuned)
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)
best_threshold_tuned = thresholds[np.argmax(f1_scores)]

probs_test_tuned = model_tuned.predict_proba(X_test)[:, 1]
preds_test_tuned = (probs_test_tuned >= best_threshold_tuned).astype(int)

print("Model A: Tuned hyperparameters + scale_pos_weight + threshold tuning ")
print(f"Threshold: {best_threshold_tuned:.3f}")
print(classification_report(y_test, preds_test_tuned, digits=4))
print(confusion_matrix(y_test, preds_test_tuned))


probs_val_smote = model_smote.predict_proba(X_val)[:, 1]
precisions_s, recalls_s, thresholds_s = precision_recall_curve(y_val, probs_val_smote)
f1_scores_s = 2 * (precisions_s * recalls_s) / (precisions_s + recalls_s + 1e-9)
best_threshold_smote = thresholds_s[np.argmax(f1_scores_s)]

probs_test_smote = model_smote.predict_proba(X_test)[:, 1]
preds_test_smote = (probs_test_smote >= best_threshold_smote).astype(int)

print("\n Model B: SMOTE + threshold tuning ")
print(f"Threshold: {best_threshold_smote:.3f}")
print(classification_report(y_test, preds_test_smote, digits=4))
print(confusion_matrix(y_test, preds_test_smote))

Model A: Tuned hyperparameters + scale_pos_weight + threshold tuning 
Threshold: 0.852
              precision    recall  f1-score   support

           0     0.9976    0.9988    0.9982       848
           1     0.9900    0.9802    0.9851       101

    accuracy                         0.9968       949
   macro avg     0.9938    0.9895    0.9917       949
weighted avg     0.9968    0.9968    0.9968       949

[[847   1]
 [  2  99]]

 Model B: SMOTE + threshold tuning 
Threshold: 0.717
              precision    recall  f1-score   support

           0     0.9976    0.9976    0.9976       848
           1     0.9802    0.9802    0.9802       101

    accuracy                         0.9958       949
   macro avg     0.9889    0.9889    0.9889       949
weighted avg     0.9958    0.9958    0.9958       949

[[846   2]
 [  2  99]]


In [12]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=RANDOM_STATE)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

model_smote = xgb.XGBClassifier(
    objective="binary:logistic", eval_metric="auc",
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.6,
    random_state=RANDOM_STATE, n_jobs=-1
)
model_smote.fit(X_train_sm, y_train_sm)

,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API <callback_api>`... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,0.6
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_method=""hist"", eval_metric=mean_absolute_error, ) reg.fit(X, y, eval_set=[(X, y)])",'auc'
,feature_types feature_types: typing.Sequence[str] | None.. versionadded:: 1.7.0Used for specifying feature types without constructing a dataframe. Seethe :py:class:`DMatrix` for details.,None


In [13]:
importances = pd.Series(model_tuned.feature_importances_, index=X_train.columns)
print(importances.sort_values(ascending=False))

Successive_Outbidding     0.836450
Winning_Ratio             0.048007
Bidding_Ratio             0.040791
Starting_Price_Average    0.021316
Bidder_Tendency           0.015330
Auction_Duration          0.014106
Auction_Bids              0.009437
Last_Bidding              0.008742
Early_Bidding             0.005822
dtype: float32


In [14]:
# Save the trained XGBoost model
model_tuned.save_model("final_xgb_shill_model.json")
print("Model saved to final_xgb_shill_model.json")

Model saved to final_xgb_shill_model.json


In [15]:
import json
from datetime import datetime

metadata = {
    "threshold": float(best_threshold_tuned),
    "feature_order": list(X_train.columns),
    "model_params": best_params,
    "scale_pos_weight": float(scale_pos_weight),
    "trained_on": datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    "test_metrics": {
        "precision_class1": 0.9900,
        "recall_class1": 0.9802,
        "f1_class1": 0.9851,
        "accuracy": 0.9968
    }
}

with open("final_model_metadata.json", "w") as f:
    json.dump(metadata, f, indent=2)

print("Metadata saved to final_model_metadata.json")

Metadata saved to final_model_metadata.json


In [16]:
import os
for fname in ["final_xgb_shill_model.json", "final_model_metadata.json"]:
    print(fname, "->", os.path.exists(fname), f"({os.path.getsize(fname)} bytes)" if os.path.exists(fname) else "")

final_xgb_shill_model.json -> True (226015 bytes)
final_model_metadata.json -> True (721 bytes)


In [17]:
# import xgboost as xgb
# import json
# import pandas as pd

# # Load model
# loaded_model = xgb.XGBClassifier()
# loaded_model.load_model("final_xgb_shill_model.json")

# # Load metadata
# with open("final_model_metadata.json") as f:
#     metadata = json.load(f)

# threshold = metadata["threshold"]
# feature_order = metadata["feature_order"]

# # Predict on new data — make sure columns match feature_order exactly
# new_data = new_data[feature_order]  # reorder/select columns to match training
# probs = loaded_model.predict_proba(new_data)[:, 1]
# predictions = (probs >= threshold).astype(int)

In [18]:
# === Experimental: test engineered features in isolation ===
df_exp = df.copy()
df_exp["bid_intensity"] = df_exp["Auction_Bids"] / (df_exp["Auction_Duration"] + 1e-5)
df_exp["price_vs_starting"] = df_exp["Winning_Ratio"] / (df_exp["Starting_Price_Average"] + 1e-5)
df_exp["tendency_x_ratio"] = df_exp["Bidder_Tendency"] * df_exp["Bidding_Ratio"]

# Quick standalone AUC check — is there any signal at all?
from sklearn.metrics import roc_auc_score
for col in ["bid_intensity", "price_vs_starting", "tendency_x_ratio"]:
    auc = roc_auc_score(df_exp["Class"], df_exp[col])
    print(f"{col}: AUC = {auc:.4f}")

bid_intensity: AUC = 0.5470
price_vs_starting: AUC = 0.7641
tendency_x_ratio: AUC = 0.9003


In [19]:
# Build feature set with the new engineered feature added
df_test = df.copy()
df_test["tendency_x_ratio"] = df_test["Bidder_Tendency"] * df_test["Bidding_Ratio"]
df_test["price_vs_starting"] = df_test["Winning_Ratio"] / (df_test["Starting_Price_Average"] + 1e-5)

id_cols = ["Record_ID", "Auction_ID", "Bidder_ID"]
df_test = df_test.drop(columns=[c for c in id_cols if c in df_test.columns])

X_new = df_test.drop(columns=["Class"])
y_new = df_test["Class"]

# Same split logic as before, same random state, for a fair comparison
X_temp2, X_test2, y_temp2, y_test2 = train_test_split(
    X_new, y_new, test_size=0.15, stratify=y_new, random_state=RANDOM_STATE
)
X_train2, X_val2, y_train2, y_val2 = train_test_split(
    X_temp2, y_temp2, test_size=0.1765, stratify=y_temp2, random_state=RANDOM_STATE
)

model_engineered = xgb.XGBClassifier(
    objective="binary:logistic", eval_metric="aucpr",
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_STATE, n_jobs=-1,
    **best_params
)
model_engineered.fit(X_train2, y_train2, eval_set=[(X_val2, y_val2)], verbose=False)

probs_val_eng = model_engineered.predict_proba(X_val2)[:, 1]
precisions_e, recalls_e, thresholds_e = precision_recall_curve(y_val2, probs_val_eng)
f1_scores_e = 2 * (precisions_e * recalls_e) / (precisions_e + recalls_e + 1e-9)
best_threshold_eng = thresholds_e[np.argmax(f1_scores_e)]

probs_test_eng = model_engineered.predict_proba(X_test2)[:, 1]
preds_test_eng = (probs_test_eng >= best_threshold_eng).astype(int)

print(" Model with engineered features ")
print(f"Threshold: {best_threshold_eng:.3f}")
print(classification_report(y_test2, preds_test_eng, digits=4))
print(confusion_matrix(y_test2, preds_test_eng))

print("\nFeature importances:")
print(pd.Series(model_engineered.feature_importances_, index=X_train2.columns).sort_values(ascending=False))

 Model with engineered features 
Threshold: 0.794
              precision    recall  f1-score   support

           0     0.9976    0.9976    0.9976       848
           1     0.9802    0.9802    0.9802       101

    accuracy                         0.9958       949
   macro avg     0.9889    0.9889    0.9889       949
weighted avg     0.9958    0.9958    0.9958       949

[[846   2]
 [  2  99]]

Feature importances:
Successive_Outbidding     0.682543
Bidding_Ratio             0.098620
Winning_Ratio             0.090372
Auction_Bids              0.043769
Auction_Duration          0.018401
tendency_x_ratio          0.018153
bid_intensity             0.015803
Bidder_Tendency           0.012415
Last_Bidding              0.008600
Early_Bidding             0.004580
Starting_Price_Average    0.003428
price_vs_starting         0.003317
dtype: float32


In [20]:
# Run this in  XGBoost notebook
X_train.to_csv("X_train.csv", index=True)
X_val.to_csv("X_val.csv", index=True)
X_test.to_csv("X_test.csv", index=True)
y_train.to_csv("y_train.csv", index=True)
y_val.to_csv("y_val.csv", index=True)
y_test.to_csv("y_test.csv", index=True)